# EDA — DKASC #38 (Q CELLS 5.9 kW mono-Si, fixed)
Run from the project root's `notebooks/` kernel (VS Code sets the notebook root to the project folder).

### [1] Load — 5-min DKASC #38 (Q CELLS 5.9 kW mono-Si, fixed). 2 corrupt lines in Aug 2025 are skipped.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
RAW = "data/raw/dkasc/59-Site_DKA-M19_C-Phase.csv"
df = pd.read_csv(RAW, on_bad_lines="skip", parse_dates=["timestamp"], index_col="timestamp").sort_index()
df = df.loc["2018-01-01":"2024-02-05"]          # the only period with power output
P, GHI, POA, TA = "Active_Power", "Global_Horizontal_Radiation", "Radiation_Global_Tilted", "Weather_Temperature_Celsius"
CAP = 5.9                                         # kW nameplate
print(len(df), df.index.min(), df.index.max())
print(df[[P, GHI, POA, TA]].describe().round(2))

### [2] Coverage: share of 5-min rows with power and with tilted irradiance, per month

In [ ]:
cov = df[[P, POA, GHI]].notna().resample("MS").mean()
fig, ax = plt.subplots(figsize=(12, 3))
cov.plot(ax=ax); ax.set(title="Monthly data coverage (share of 5-min rows present)", ylim=(0, 1.05))
plt.tight_layout(); plt.savefig("results/dkasc_eda/fig_coverage.png", dpi=110)

### [3] Daily energy vs daily tilted insolation, and a performance ratio (PR) over time

In [ ]:
d = pd.DataFrame({"kWh": df[P].clip(lower=0).resample("D").sum() * 5/60,
                  "poa_kWh_m2": df[POA].clip(lower=0).resample("D").sum() * 5/60/1000,
                  "n": df[P].resample("D").count()})
d = d[d.n > 250]                                  # keep days with >~87% of rows
d["PR"] = d.kWh / (d.poa_kWh_m2 * CAP)
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(d.kWh, lw=.6); axes[0].set_ylabel("kWh/day")
axes[1].plot(d.poa_kWh_m2, lw=.6, color="orange"); axes[1].set_ylabel("POA kWh/m²/day")
axes[2].plot(d.PR, lw=.4, alpha=.5); axes[2].plot(d.PR.rolling(30, center=True).median(), color="k")
axes[2].set(ylabel="PR", ylim=(0.4, 1.1)); axes[0].set_title("Daily energy, insolation and performance ratio")
plt.tight_layout(); plt.savefig("results/dkasc_eda/fig_daily.png", dpi=110)
print("PR yearly median:\n", d.PR.groupby(d.index.year).median().round(3))

### [4] Power vs tilted irradiance (the core physics) — one panel per year, colour = ambient temp

In [ ]:
day = df[(df[POA] > 20)].dropna(subset=[P, POA])
yrs = sorted(day.index.year.unique())
fig, axes = plt.subplots(1, len(yrs), figsize=(3 * len(yrs), 3.2), sharey=True)
for ax, y in zip(axes, yrs):
    s = day[day.index.year == y].sample(min(6000, (day.index.year == y).sum()), random_state=0)
    ax.scatter(s[POA], s[P], c=s[TA], s=1, cmap="coolwarm", vmin=5, vmax=40)
    ax.plot([0, 1200], [0, CAP * 1.2], "k--", lw=.6); ax.set(title=str(y), xlabel="POA W/m²")
axes[0].set_ylabel("kW"); plt.tight_layout(); plt.savefig("results/dkasc_eda/fig_scatter.png", dpi=110)

### [5] Timing check: power vs tilted irradiance on one clear and one cloudy day (timestamp alignment)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
ok = d[d.poa_kWh_m2 > 0.5]                       # skip days where the tilted sensor was down
for ax, day_ in zip(axes, [ok[ok.poa_kWh_m2 > ok.poa_kWh_m2.quantile(.9)].index[100].date(),
                           ok[ok.poa_kWh_m2 < ok.poa_kWh_m2.quantile(.15)].index[50].date()]):
    x = df.loc[str(day_)]
    ax.plot(x.index, x[P], label="kW"); ax2 = ax.twinx(); ax2.plot(x.index, x[POA], color="orange", label="POA")
    ax.set_title(str(day_)); ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig("results/dkasc_eda/fig_days.png", dpi=110)

### [6] Sensor glitches: tilted-sensor outages (POA≈0 while power is normal) and impossible temperatures

In [ ]:
bad_poa = d[(d.poa_kWh_m2 < 0.5) & (d.kWh > 15)]
print("days with POA sensor down but output normal:", len(bad_poa), bad_poa.index.min().date(), "->", bad_poa.index.max().date())
print("temperature < -5 °C or > 50 °C rows:", ((df[TA] < -5) | (df[TA] > 50)).sum())

### [7] Ramps: distribution of 15-min and 60-min changes in power (normalised by capacity), daytime only

In [ ]:
p15 = df[P].resample("15min").mean(); p60 = df[P].resample("60min").mean()
r15 = p15.diff().abs() / CAP; r60 = p60.diff().abs() / CAP
r15 = r15[p15.shift(1) > 0.05 * CAP]; r60 = r60[p60.shift(1) > 0.05 * CAP]
for name, r in [("15-min", r15), ("60-min", r60)]:
    print(f"{name} |ΔP|/cap: p50 {r.median():.3f}  p95 {r.quantile(.95):.3f}  >0.3: {(r > .3).sum()}  >0.5: {(r > .5).sum()}")